In [2]:
import pandas as pd
import joblib
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
import idx2numpy

In [4]:
already_generate_idx = False
already_generate_idx = True

if not already_generate_idx:

    
    """
    This runs only one time, to pre-process all the model and dataset
    
    Load original data set as "dataset"
    Mapping the ego_a (acceleration) to 3 classes [0,1,2]
    
    ========
    0 -> break
    1 -> idle
    2 -> speed up
    ========
    
    split the dataset into training-datas.idx and training-lables.idx,
    for the use of vehicle and marabou
    
    """

    # load dataset
    dataset = pd.read_csv('simulation-dataset.csv')
    print(f"Processing dataset: {dataset.shape}...")
    
    # rule of mapping labels
    def classify_acceleration(a, threshold=0.05):
        if a > threshold:
            return 2  # speed up
        elif a < -threshold:
            return 0  # break
        else:
            return 1  # idle
    
    # transfer the acceleration from -5, 0, 3 to class 0,1,2
    dataset['action'] = dataset['ego_a'].apply(classify_acceleration)
    print(dataset['action'].value_counts())
    
    #define model input and output
    FEATURE_COLUMNS = ['d_front', 'd_back', 'v_front', 'v_back']
    LABEL_COLUMN = 'action'
    
    training_datas_unnormalised = dataset[FEATURE_COLUMNS].values
    training_labels= dataset[LABEL_COLUMN].values
    
    # normalise input datas
    scaler = joblib.load('scaler.pkl')
    training_datas = scaler.transform(training_datas_unnormalised)
    
    # transfer and save as .ixd
    datas_to_save = training_datas.astype(np.float32)
    labels_to_save = training_labels.astype(np.int32)
    
    idx2numpy.convert_to_file('trainingDatas.idx', datas_to_save)
    idx2numpy.convert_to_file('trainingLabes.idx', labels_to_save)

    print("\n ...Finished")


Processing dataset: (8006, 7)...
action
1    7601
2     250
0     155
Name: count, dtype: int64

 ...Finished


In [8]:
feature_file = 'trainingDatas.idx'
label_file = 'trainingLabes.idx'



features_array = idx2numpy.convert_from_file(feature_file)
labels_array = idx2numpy.convert_from_file(label_file)

print("Features:")
print(f"data type: {features_array.dtype}")
print(f"data shape: {features_array.shape}")
print("\nFirst 5 lines:")
print(features_array[:5])


print("\nLabels")
print(f"datatype: {labels_array.dtype}")
print(f"datashape: {labels_array.shape}")
print("\nFirst 50 labels:")
print(labels_array[:50])


    

Features:
data type: >f4
data shape: (8006, 4)

First 5 lines:
[[-0.44419482 -0.8948045  -0.18360142 -0.3625577 ]
 [-0.4445358  -0.89436007 -0.3495427  -0.12819463]
 [-0.44521976 -0.8929511  -0.35052216  0.14612773]
 [-0.44591737 -0.89053065 -0.3561972   0.40518206]
 [-0.4466401  -0.8876108  -0.362737    0.40946245]]

Labels
datatype: >i4
datashape: (8006,)

First 50 labels:
[1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 0 0 0 0 1 1 1]
